# Data extraction

This notebook uses BigQuery to retrieve processed public health data from Base dos Dados.

The query combines two datasets:

- **Outpatient Information System (SIA)**, used to identify outpatient records related to gambling and betting;
- **National Registry of Health Establishments (CNES)**, used to obtain information about the healthcare facilities where the visits were recorded.

The analysis includes records from **2015 to 2025** associated with the following ICD-10 codes:

- **F63.0** — Pathological gambling
- **Z72.6** — Gambling and betting habits

The query also joins geographic reference tables from Base dos Dados to retrieve the names of states and municipalities.

## Data sources

- [SIA — Outpatient production](https://basedosdados.org/dataset/22d1f0d6-9bbc-4653-a841-7734867d2319?table=5613bffd-f741-4f74-a48e-685d6438f354)
- CNES — National Registry of Health Establishments

In [8]:
import basedosdados as bd
import os
from dotenv import load_dotenv
load_dotenv()
import pandas as pd

In [9]:
billing_id = os.getenv("billing_id")

query = """
WITH atendimentos AS (
    SELECT
        dados.ano,
        dados.mes,
        dados.sigla_uf,

        diretorio_uf.nome AS nome_uf,

        dados.id_municipio,
        diretorio_municipio.nome AS nome_municipio,

        dados.id_estabelecimento_cnes,

        dados.cid_principal_subcategoria,
        dados.cid_secundario_subcategoria,
        dados.cid_causas_associadas_subcategoria,

        dados.sexo_paciente,
        dados.idade_paciente,
        dados.raca_cor_paciente,

        dados.indicador_uf_residencia_paciente,
        dados.indicador_municipio_residencia_paciente

    FROM `basedosdados.br_ms_sia.producao_ambulatorial` AS dados

    LEFT JOIN (
        SELECT DISTINCT
            sigla,
            nome
        FROM `basedosdados.br_bd_diretorios_brasil.uf`
    ) AS diretorio_uf
        ON dados.sigla_uf = diretorio_uf.sigla

    LEFT JOIN (
        SELECT DISTINCT
            id_municipio,
            nome
        FROM `basedosdados.br_bd_diretorios_brasil.municipio`
    ) AS diretorio_municipio
        ON dados.id_municipio = diretorio_municipio.id_municipio

    WHERE dados.ano BETWEEN 2015 AND 2025
      AND (
          dados.cid_principal_subcategoria IN ('F630', 'Z726')
          OR dados.cid_secundario_subcategoria IN ('F630', 'Z726')
          OR dados.cid_causas_associadas_subcategoria IN ('F630', 'Z726')
      )
),

cnes AS (
    SELECT
        ano,
        mes,
        id_estabelecimento_cnes,
        id_municipio AS id_municipio_cnes,
        cep,
        indicador_vinculo_sus

    FROM `basedosdados.br_ms_cnes.estabelecimento`

    WHERE ano BETWEEN 2015 AND 2025
)

SELECT
    atendimentos.*,

    cnes.id_municipio_cnes,

    diretorio_municipio_cnes.nome AS nome_municipio_cnes,

    cnes.cep,
    cnes.indicador_vinculo_sus

FROM atendimentos

LEFT JOIN cnes
    ON atendimentos.ano = cnes.ano
    AND atendimentos.mes = cnes.mes
    AND atendimentos.id_estabelecimento_cnes = cnes.id_estabelecimento_cnes

LEFT JOIN (
    SELECT DISTINCT
        id_municipio,
        nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
) AS diretorio_municipio_cnes
    ON cnes.id_municipio_cnes = diretorio_municipio_cnes.id_municipio
"""

df = bd.read_sql(
    query=query,
    billing_project_id=billing_id
)

df

Downloading: 100%|██████████|


,ano,mes,sigla_uf,nome_uf,id_municipio,nome_municipio,id_estabelecimento_cnes,cid_principal_subcategoria,cid_secundario_subcategoria,cid_causas_associadas_subcategoria,sexo_paciente,idade_paciente,raca_cor_paciente,indicador_uf_residencia_paciente,indicador_municipio_residencia_paciente,id_municipio_cnes,nome_municipio_cnes,cep,indicador_vinculo_sus
0,2024,5,SP,São Paulo,3550308,São Paulo,2078015,F630,None,None,M,28,1,0,1,3550308,São Paulo,05403010,1
1,2022,10,MA,Maranhão,2106755,Miranda do Norte,6569862,F630,None,None,M,21,3,0,0,2106755,Miranda do Norte,65495000,1
2,2022,9,BA,Bahia,2905701,Camaçari,7781555,F630,None,None,M,26,3,0,0,2905701,Camaçari,42809382,1
3,2024,10,SC,Santa Catarina,4209003,Joaçaba,3757412,None,None,F630,M,35,3,0,0,4209003,Joaçaba,89600000,1
4,2024,12,RO,Rondônia,1100288,Rolim de Moura,7217765,F630,None,None,F,64,3,0,0,1100288,Rolim de Moura,76940000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7438,2019,8,RS,Rio Grande do Sul,4318804,São Lourenço do Sul,3019799,None,None,F630,M,58,1,0,0,4318804,São Lourenço do Sul,96170000,1
7439,2024,8,SP,São Paulo,3520509,Indaiatuba,3762092,F630,None,None,M,23,3,0,0,3520509,Indaiatuba,13334085,1
7440,2024,1,SP,São Paulo,3556206,Valinhos,6753205,F630,None,None,M,36,1,0,0,3556206,Valinhos,13276045,1
7441,2017,9,RN,Rio Grande do Norte,2404200,Goianinha,6464343,F630,None,None,F,56,3,0,1,2404200,Goianinha,59173000,1


In [10]:
df.columns

Index(['ano', 'mes', 'sigla_uf', 'nome_uf', 'id_municipio', 'nome_municipio',
       'id_estabelecimento_cnes', 'cid_principal_subcategoria',
       'cid_secundario_subcategoria', 'cid_causas_associadas_subcategoria',
       'sexo_paciente', 'idade_paciente', 'raca_cor_paciente',
       'indicador_uf_residencia_paciente',
       'indicador_municipio_residencia_paciente', 'id_municipio_cnes',
       'nome_municipio_cnes', 'cep', 'indicador_vinculo_sus'],
      dtype='object')

In [11]:
df.shape

(7443, 19)

In [12]:
df.head()

,ano,mes,sigla_uf,nome_uf,id_municipio,nome_municipio,id_estabelecimento_cnes,cid_principal_subcategoria,cid_secundario_subcategoria,cid_causas_associadas_subcategoria,sexo_paciente,idade_paciente,raca_cor_paciente,indicador_uf_residencia_paciente,indicador_municipio_residencia_paciente,id_municipio_cnes,nome_municipio_cnes,cep,indicador_vinculo_sus
0,2024,5,SP,São Paulo,3550308,São Paulo,2078015,F630,None,None,M,28,1,0,1,3550308,São Paulo,05403010,1
1,2022,10,MA,Maranhão,2106755,Miranda do Norte,6569862,F630,None,None,M,21,3,0,0,2106755,Miranda do Norte,65495000,1
2,2022,9,BA,Bahia,2905701,Camaçari,7781555,F630,None,None,M,26,3,0,0,2905701,Camaçari,42809382,1
3,2024,10,SC,Santa Catarina,4209003,Joaçaba,3757412,None,None,F630,M,35,3,0,0,4209003,Joaçaba,89600000,1
4,2024,12,RO,Rondônia,1100288,Rolim de Moura,7217765,F630,None,None,F,64,3,0,0,1100288,Rolim de Moura,76940000,1


In [13]:
df["indicador_uf_residencia_paciente"].unique()

<IntegerArray>
[0, 1]
Length: 2, dtype: Int64

In [14]:
df.to_csv("sia-data.csv", index=False)